## BGE Robustness analysis

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT=Path("..")
BGE=ROOT/"results"/"week13"/"bge_confirmatory"
OUT=BGE/"analysis"
OUT.mkdir(exist_ok=True)

POWER=BGE/"bge_power_runs.csv"
NULL=BGE/"bge_null_runs.csv"
MANIFEST=BGE/"bge_execution_manifest.csv"

In [3]:
# load
def read_csv(p):
    return pd.read_csv(p,keep_default_na=False,na_values=[""])

power=read_csv(POWER)
null=read_csv(NULL)
manifest=read_csv(MANIFEST)

print("Power:",len(power))
print("Null:",len(null))
print("Total:",len(power)+len(null))
print("Manifest:",len(manifest))

Power: 6000
Null: 1000
Total: 7000
Manifest: 7000


In [4]:
# final integrity check
def basic(df,name):
    assert df["run_id"].is_unique,f"{name}: duplicate run_id"
    assert df["p_value"].notna().all(),f"{name}: missing p"
    assert np.isfinite(df["p_value"]).all(),f"{name}: non-finite p"
    assert df["p_value"].between(0,1).all(),f"{name}: invalid p"
    assert df["reject"].isin([0,1]).all(),f"{name}: invalid reject"
    assert np.isfinite(df["runtime"]).all(),f"{name}: invalid runtime"

basic(power,"POWER"); basic(null,"NULL")

assert len(power)==6000
assert len(null)==1000
assert power["kind"].eq("power").all()
assert null["kind"].eq("null").all()

pc=power.groupby(["scenario","severity","test"]).size()
nc=null.groupby(["test"]).size()

assert len(pc)==30 and (pc==200).all()
assert len(nc)==5 and (nc==200).all()

expected=set(range(200))
for _,g in power.groupby(["scenario","severity","test"]):
    assert set(g["rep"])==expected
for _,g in null.groupby(["test"]):
    assert set(g["rep"])==expected

result_ids=set(power.run_id)|set(null.run_id)
assert result_ids==set(manifest.run_id)

print("BGE FINAL INTEGRITY CHECK PASSED")
print("Power cells:",len(pc))
print("Null cells:",len(nc))
print("Formal runs:",len(result_ids))

BGE FINAL INTEGRITY CHECK PASSED
Power cells: 30
Null cells: 5
Formal runs: 7000


In [5]:
# save integrity record 
integrity={
    "power_rows":len(power),
    "null_rows":len(null),
    "total_rows":len(power)+len(null),
    "power_cells":len(pc),
    "null_cells":len(nc),
    "repetitions_per_cell":200,
    "integrity_passed":True
}

with open(OUT/"integrity_report.json","w") as f:
    json.dump(integrity,f,indent=2)

integrity

{'power_rows': 6000,
 'null_rows': 1000,
 'total_rows': 7000,
 'power_cells': 30,
 'null_cells': 5,
 'repetitions_per_cell': 200,
 'integrity_passed': True}

In [6]:
# wilson confidence interval helper 
def wilson(k,n,z=1.959963984540054):
    p=k/n; z2=z*z; d=1+z2/n
    c=(p+z2/(2*n))/d
    h=z*np.sqrt(p*(1-p)/n+z2/(4*n*n))/d
    return c-h,c+h

In [7]:
# BGE type1 calibration 
type1=(
    null.groupby("test",as_index=False)
    .agg(
        rejections=("reject","sum"),
        n_reps=("reject","size"),
        mean_p=("p_value","mean"),
        median_p=("p_value","median"),
        mean_runtime=("runtime","mean")
    )
)

type1["type_I_error"]=type1.rejections/type1.n_reps

ci=type1.apply(lambda r:wilson(r.rejections,r.n_reps),axis=1)
type1["wilson_low"]=[x[0] for x in ci]
type1["wilson_high"]=[x[1] for x in ci]

type1["calibration_status"]=np.where(
    type1.rejections.between(4,16),
    "within preregistered range",
    "FLAG"
)

type1.round(4)

,test,rejections,n_reps,mean_p,median_p,mean_runtime,type_I_error,wilson_low,wilson_high,calibration_status
0,C2ST-Linear,12,200,0.5470,0.5644,0.0104,0.060,0.0347,0.1019,within preregistered range
1,C2ST-RBF,10,200,0.5536,0.5644,0.0139,0.050,0.0274,0.0896,within preregistered range
2,Energy,9,200,0.5068,0.5248,0.0310,0.045,0.0239,0.0833,within preregistered range
3,MMD,8,200,0.5075,0.5396,0.0297,0.040,0.0204,0.0769,within preregistered range
4,Sinkhorn,7,200,0.5152,0.5248,0.9476,0.035,0.0171,0.0705,within preregistered range


In [8]:
# flags 
flags=type1[type1.calibration_status=="FLAG"]

print("Within range:",
      (type1.calibration_status=="within preregistered range").sum())
print("Flagged:",len(flags))
print("Total:",len(type1))

flags

Within range: 5
Flagged: 0
Total: 5


,test,rejections,n_reps,mean_p,median_p,mean_runtime,type_I_error,wilson_low,wilson_high,calibration_status


The formal BGE calibration passed completely: 5/5 configurations fell within the pre-registered range, with 0 flags.

This means that, in a "clean-vs-clean" scenario, none of the five tests exhibited a concerning false-positive rate. Specifically:
- C2ST-Linear: 0.060
- C2ST-RBF: 0.050
- Energy: 0.045
- MMD: 0.040
- Sinkhorn: 0.035
All of these fall within our pre-registered range of 0.020–0.080. Furthermore, the Wilson intervals are consistent with reasonable calibration around α=.05. Consequently, we do not encounter the conservative-bias issue here that we observed in four of the primary configurations.

This is significant because, should BGE now show low or high power, we can interpret this with much greater confidence as the test’s actual behavior under that representation—rather than as an obvious artifact of poor calibration.

In [9]:
# save type 1
type1.to_csv(OUT/"bge_type1_wilson.csv",index=False)
flags.to_csv(OUT/"bge_type1_flags.csv",index=False)

In [10]:
# BGE power + CIs
pwr=(
    power.groupby(["scenario","severity","test"],as_index=False)
    .agg(
        rejections=("reject","sum"),
        n_reps=("reject","size"),
        mean_p=("p_value","mean"),
        median_p=("p_value","median"),
        mean_runtime=("runtime","mean")
    )
)

pwr["power"]=pwr.rejections/pwr.n_reps

ci=pwr.apply(lambda r:wilson(r.rejections,r.n_reps),axis=1)
pwr["wilson_low"]=[x[0] for x in ci]
pwr["wilson_high"]=[x[1] for x in ci]

pwr=pwr.sort_values(["scenario","severity","test"]).reset_index(drop=True)
pwr

,scenario,severity,test,rejections,n_reps,mean_p,median_p,mean_runtime,power,wilson_low,wilson_high
0,Entity Inconsistency,0.1,C2ST-Linear,1,200,0.746931,1.000000,0.007637,0.005,8.831687e-04,0.027774
1,Entity Inconsistency,0.1,C2ST-RBF,0,200,0.686089,0.599010,0.011111,0.000,1.734723e-18,0.018845
2,Entity Inconsistency,0.1,Energy,171,200,0.034307,0.009901,0.031290,0.855,7.995133e-01,0.897106
3,Entity Inconsistency,0.1,MMD,170,200,0.035941,0.019802,0.026615,0.850,7.939442e-01,0.892864
4,Entity Inconsistency,0.1,Sinkhorn,2,200,0.616238,0.653465,0.884577,0.010,2.746658e-03,0.035722
5,Entity Inconsistency,0.3,C2ST-Linear,58,200,0.214653,0.138614,0.011424,0.290,2.315403e-01,0.356375
6,Entity Inconsistency,0.3,C2ST-RBF,51,200,0.202673,0.128713,0.014049,0.255,1.996050e-01,0.319629
7,Entity Inconsistency,0.3,Energy,200,200,0.009901,0.009901,0.039181,1.000,9.811547e-01,1.000000
8,Entity Inconsistency,0.3,MMD,200,200,0.009901,0.009901,0.035475,1.000,9.811547e-01,1.000000
9,Entity Inconsistency,0.3,Sinkhorn,4,200,0.751782,0.866337,0.859775,0.020,7.804426e-03,0.050287


In [11]:
# attatch null calibration 
cal=type1[
    ["test","type_I_error","wilson_low","wilson_high","calibration_status"]
].rename(columns={
    "wilson_low":"type1_wilson_low",
    "wilson_high":"type1_wilson_high"
})

bge=pwr.merge(cal,on="test",how="left",validate="many_to_one")

assert len(bge)==30
bge.to_csv(OUT/"bge_power_with_calibration.csv",index=False)

bge

,scenario,severity,test,rejections,n_reps,mean_p,median_p,mean_runtime,power,wilson_low,wilson_high,type_I_error,type1_wilson_low,type1_wilson_high,calibration_status
0,Entity Inconsistency,0.1,C2ST-Linear,1,200,0.746931,1.000000,0.007637,0.005,8.831687e-04,0.027774,0.060,0.034652,0.101932,within preregistered range
1,Entity Inconsistency,0.1,C2ST-RBF,0,200,0.686089,0.599010,0.011111,0.000,1.734723e-18,0.018845,0.050,0.027383,0.089578,within preregistered range
2,Entity Inconsistency,0.1,Energy,171,200,0.034307,0.009901,0.031290,0.855,7.995133e-01,0.897106,0.045,0.023853,0.083297,within preregistered range
3,Entity Inconsistency,0.1,MMD,170,200,0.035941,0.019802,0.026615,0.850,7.939442e-01,0.892864,0.040,0.020406,0.076932,within preregistered range
4,Entity Inconsistency,0.1,Sinkhorn,2,200,0.616238,0.653465,0.884577,0.010,2.746658e-03,0.035722,0.035,0.017056,0.070471,within preregistered range
5,Entity Inconsistency,0.3,C2ST-Linear,58,200,0.214653,0.138614,0.011424,0.290,2.315403e-01,0.356375,0.060,0.034652,0.101932,within preregistered range
6,Entity Inconsistency,0.3,C2ST-RBF,51,200,0.202673,0.128713,0.014049,0.255,1.996050e-01,0.319629,0.050,0.027383,0.089578,within preregistered range
7,Entity Inconsistency,0.3,Energy,200,200,0.009901,0.009901,0.039181,1.000,9.811547e-01,1.000000,0.045,0.023853,0.083297,within preregistered range
8,Entity Inconsistency,0.3,MMD,200,200,0.009901,0.009901,0.035475,1.000,9.811547e-01,1.000000,0.040,0.020406,0.076932,within preregistered range
9,Entity Inconsistency,0.3,Sinkhorn,4,200,0.751782,0.866337,0.859775,0.020,7.804426e-03,0.050287,0.035,0.017056,0.070471,within preregistered range


In [12]:
# power matrix 
bge_matrix=bge.pivot(
    index=["scenario","severity"],
    columns="test",
    values="power"
)

bge_matrix.round(3)

test                                      C2ST-Linear  C2ST-RBF  Energy  \
scenario                        severity                                  
Entity Inconsistency            0.1             0.005     0.000   0.855   
                                0.3             0.290     0.255   1.000   
                                0.6             0.905     0.900   1.000   
Missing Cross-Channel Reference 0.1             0.000     0.005   0.085   
                                0.3             0.060     0.135   0.470   
                                0.6             0.535     0.455   0.985   

test                                        MMD  Sinkhorn  
scenario                        severity                   
Entity Inconsistency            0.1       0.850     0.010  
                                0.3       1.000     0.020  
                                0.6       1.000     0.025  
Missing Cross-Channel Reference 0.1       0.085     0.010  
                                0.3       0.480     0.010  
                                0.6       0.985     0.170

**The general pattern is indeed robust to a change of encoder, but some specific blind spots depend quite heavily on the representation.**

Energy and MMD are extremely stable as a pair. When used with BGE, they remain virtually identical to each other under all conditions. For the "Entity" case, they even show marked improvement compared to MiniLM; for the "Missing" case, their performance dips slightly at low/medium severity levels, but at 60% prevalence, they still perform essentially at the same level as "power 1." This strongly reinforces the conclusion that their robust performance was not merely a quirk specific to MiniLM.

The "blind spot" of C2ST for the "Entity" case is not entirely encoder-agnostic. At 10% prevalence, it remains stark: BGE yields .005/.000 for Linear/RBF kernels, whereas Energy/MMD hover around .85. Thus, the blind spot regarding subtle defects persists. However, as prevalence increases, BGE changes the picture entirely: at 60%, the values ​​are .905 (Linear) and .900 (RBF), compared to approximately .165/.180 when using MiniLM.

**C2ST sensitivity to Entity Inconsistency is strongly representation-dependent. The low-severity blind spot persisted under BGE, but higher-severity detection improved substantially relative to MiniLM.**

**Fidelity response depends on the interaction between defect, representation and metric/test configuration.**


With Missing Cross-Channel is different. In this case, swapping MiniLM for BGE does not significantly alter C2ST or Energy/MMD. Energy/MMD maintain the same general pattern: weak/moderate → moderate → nearly perfect as prevalence increases. This appears quite robust. However, Sinkhorn changes drastically. With MiniLM, the values ​​went from .125 → .310 → .685; with BGE, they drop to:
.010 -> .010 -> .170


Sinkhorn appears sensitive to the geometry induced by the encoder. And since its BGE null is perfectly calibrated (Type I = 0.035), we cannot explain this result by claiming that the test was simply broken or poorly calibrated.

In [13]:
# runtime 
runtime=(
    pd.concat([
        power.assign(source="power"),
        null.assign(source="null")
    ])
    .groupby(["source","test"],as_index=False)
    .runtime.agg(["mean","median"])
)

runtime.round(4)

,source,test,mean,median
0,null,C2ST-Linear,0.0104,0.0075
1,null,C2ST-RBF,0.0139,0.0120
2,null,Energy,0.0310,0.0300
3,null,MMD,0.0297,0.0300
4,null,Sinkhorn,0.9476,0.8602
5,power,C2ST-Linear,0.0118,0.0106
6,power,C2ST-RBF,0.0146,0.0136
7,power,Energy,0.0344,0.0330
8,power,MMD,0.0346,0.0327
9,power,Sinkhorn,0.8806,0.8160


#### Preliminary conclusion
The BGE robustness analysis preserved the broad advantage of MMD and Energy Distance, particularly for low-severity Entity Inconsistency and moderate-to-high Missing Cross-Channel Reference. However, changing the encoder materially altered the behavior of C2ST and Sinkhorn. C2ST remained nearly powerless for subtle Entity Inconsistency at 10% prevalence but became substantially more sensitive at higher prevalence under BGE than under MiniLM. Conversely, Sinkhorn became markedly less sensitive to Missing Cross-Channel Reference under BGE. These results indicate that metric blind spots are not solely properties of the statistical test; they can emerge from interactions between the defect, representation geometry, and test configuration.

In [14]:
# reproducible comparison 
minilm={
("Entity Inconsistency",.1):[0,.0,.605,.640,.045],
("Entity Inconsistency",.3):[.025,.015,1,1,.025],
("Entity Inconsistency",.6):[.165,.180,1,1,.090],
("Missing Cross-Channel Reference",.1):[.005,0,.140,.140,.125],
("Missing Cross-Channel Reference",.3):[.125,.080,.550,.560,.310],
("Missing Cross-Channel Reference",.6):[.505,.515,.995,.995,.685]
}

tests=["C2ST-Linear","C2ST-RBF","Energy","MMD","Sinkhorn"]

rows=[]
for (scenario,severity),vals in minilm.items():
    for test,power0 in zip(tests,vals):
        rows.append({
            "scenario":scenario,
            "severity":severity,
            "test":test,
            "minilm_power":power0
        })

minilm=pd.DataFrame(rows)

comparison=bge[
    ["scenario","severity","test","power"]
].rename(columns={"power":"bge_power"}).merge(
    minilm,
    on=["scenario","severity","test"],
    validate="one_to_one"
)

comparison["delta_bge_minus_minilm"]=(
    comparison.bge_power-comparison.minilm_power
)

comparison.sort_values(
    ["scenario","severity","test"]
).round(3)

,scenario,severity,test,bge_power,minilm_power,delta_bge_minus_minilm
0,Entity Inconsistency,0.1,C2ST-Linear,0.005,0.000,0.005
1,Entity Inconsistency,0.1,C2ST-RBF,0.000,0.000,0.000
2,Entity Inconsistency,0.1,Energy,0.855,0.605,0.250
3,Entity Inconsistency,0.1,MMD,0.850,0.640,0.210
4,Entity Inconsistency,0.1,Sinkhorn,0.010,0.045,-0.035
5,Entity Inconsistency,0.3,C2ST-Linear,0.290,0.025,0.265
6,Entity Inconsistency,0.3,C2ST-RBF,0.255,0.015,0.240
7,Entity Inconsistency,0.3,Energy,1.000,1.000,0.000
8,Entity Inconsistency,0.3,MMD,1.000,1.000,0.000
9,Entity Inconsistency,0.3,Sinkhorn,0.020,0.025,-0.005


In [15]:
comparison.to_csv(
    OUT/"bge_vs_minilm_n100.csv",
    index=False
)